# Instagram Engagement Prediction Using Machine Learning
**Course:** Predictive Analysis  
**Approach:** Supervised Learning – Regression  
**Algorithm:** Random Forest Regressor  

> **Dataset note:** This project uses a synthetic/demo dataset generated for coursework. It is not actual Instagram Insights data and results should not be interpreted as real-world performance guarantees.


## 1. Problem Statement
Predict the number of likes on an Instagram post using available post details such as follower count, content type, posting time, day of week, hashtag count and Reel duration.

## 2. Objectives
- Explore and visualize post-performance data.
- Prepare input features for machine learning.
- Train a Random Forest Regression model.
- Evaluate predictions using MAE, RMSE and R².
- Predict likes for a new example post.


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()


## 4. Load Dataset

In [ ]:
# Upload instagram_engagement_synthetic.csv to Colab, or keep it in the same folder
df = pd.read_csv("instagram_engagement_synthetic.csv")
print("Dataset shape:", df.shape)
display(df.head())
print(df.info())


## 5. Exploratory Data Analysis

In [ ]:
print(df.describe(include="all"))
print("\\nMissing values:\\n", df.isnull().sum())


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df["likes"], kde=True)
plt.title("Distribution of Instagram Likes")
plt.xlabel("Likes")
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="post_type", y="likes")
plt.title("Likes by Content Type")
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
sns.scatterplot(data=df, x="followers", y="likes", hue="post_type")
plt.title("Followers vs Likes")
plt.show()


## 6. Data Preparation and Model Training
The target variable is `likes`. We exclude comments, saves, shares and engagement rate from the inputs because these are engagement outcomes and would cause target leakage when predicting likes.

In [ ]:
target = "likes"
features = ["followers", "post_type", "posting_hour", "day_of_week",
            "hashtag_count", "reel_duration_sec"]

X = df[features]
y = df[target]

numeric_features = ["followers", "posting_hour", "hashtag_count", "reel_duration_sec"]
categorical_features = ["post_type", "day_of_week"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=150, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
print("Model training completed.")


## 7. Model Evaluation

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.2f} likes")
print(f"RMSE: {rmse:.2f} likes")
print(f"R² Score: {r2:.3f}")


In [ ]:
results = pd.DataFrame({"Actual Likes": y_test.values,
                         "Predicted Likes": np.round(predictions).astype(int)})
display(results.head(10))

plt.figure(figsize=(6,6))
plt.scatter(y_test, predictions, alpha=0.7)
plt.xlabel("Actual Likes")
plt.ylabel("Predicted Likes")
plt.title("Actual vs Predicted Likes")
plt.show()


## 8. Predict Engagement for a New Post

In [ ]:
new_post = pd.DataFrame([{
    "followers": 12000,
    "post_type": "Reel",
    "posting_hour": 19,
    "day_of_week": "Saturday",
    "hashtag_count": 8,
    "reel_duration_sec": 30
}])

predicted_likes = model.predict(new_post)[0]
print(f"Estimated likes for the sample post: {predicted_likes:.0f}")


## 9. Conclusion
A Random Forest Regression model was trained to estimate Instagram post likes from post characteristics. The model was evaluated on a held-out test set using MAE, RMSE and R². The notebook demonstrates the complete predictive analysis workflow: data loading, exploration, preprocessing, training, evaluation and prediction.

**Limitations:** The dataset is synthetic and designed only to demonstrate the method. Real Instagram performance depends on many additional factors, and this model should be retrained and evaluated on genuine, permissioned historical data before practical use.
